# Deploying NVIDIA Nemotron-3-Super with SGLang

This notebook will walk you through how to run the `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B` model with SGLang.

[SGLang](https://github.com/sgl-project/sglang) is a fast serving framework for large language models and vision language models.

For more details on the model [click here](https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16)

Prerequisites:
- NVIDIA GPU with recent drivers (>= 264 GB VRAM for BF16, >= 160 GB VRAM for FP8, >= 80 GB VRAM for NVFP4) and CUDA 12.x.
- Python 3.10+

Note: NVFP4 requires Blackwell architecture. This notebook includes B200-focused steps for NVFP4 setup and serving.

#### Launch on NVIDIA Brev
You can simplify the environment setup by using [NVIDIA Brev](https://developer.nvidia.com/brev). Click the button to launch this project on a Brev instance with the necessary dependencies pre-configured.

Once deployed, click on the "Open Notebook" button to get started with this guide. 

**For BF16 (4x H100):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-39d03Y3mDAiGuIrnHKmZwv0tA4s)

**For FP8 (2x H100):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-3Ajvh1M8JGbE4UIa0YYmk7hSgeD)

## Install dependencies

For FP8, ensure CUDA toolkit and build tools are available in your environment before first run:

```shell
sudo apt update
sudo apt install -y cuda-toolkit-12-8 ninja-build gcc g++ build-essential
export CUDA_HOME=/usr/local/cuda-12.8
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib64:${LD_LIBRARY_PATH}
which nvcc
nvcc --version
```

FP8 may take longer on first startup due to kernel compile/autotune; later runs are usually faster from cache.

In [1]:
#If pip not found
!python -m ensurepip --default-pip

Looking in links: /tmp/tmpqdt1j2kl
Processing /tmp/tmpqdt1j2kl/pip-25.0.1-py3-none-any.whl


In [ ]:
%pip install sglang==0.5.9 torch==2.9.1

## Verify GPU

Confirm CUDA is available and your GPU is visible to PyTorch.

In [2]:
# GPU environment check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")

CUDA available: True
Num GPUs: 4
GPU[0]: NVIDIA H100 PCIe
GPU[1]: NVIDIA H100 PCIe
GPU[2]: NVIDIA H100 PCIe
GPU[3]: NVIDIA H100 PCIe


## Start SGLang Server

SGLang runs as a separate server process.

Before starting the server, ensure that your notebook and terminal are in the same virtual environment.

For example, on Brev run:
```shell
source /home/shadeform/.venv/bin/activate
```

If you are not on Brev, replace this with your own virtual environment path.

Choose the variant you want to serve:
- BF16: `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16` on 4x H100 (`--tp 4`, `--ep 1`)
- FP8: `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8` on 2x H100 (`--tp 2`, `--ep 1`)
- NVFP4: `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-NVFP4` on B200 (`--tp 1`, `--ep 1`)

Then, run the matching command in the terminal.

### BF16 (4x H100)

```shell
python3 -m sglang.launch_server \
  --model-path nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16 \
  --host 0.0.0.0 \
  --port 5000 \
  --trust-remote-code \
  --tp 4 \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nano_v3
```

### FP8 (2x H100)

Note: the first FP8 startup may take longer than BF16 while kernels are compiled/autotuned.

```shell
python3 -m sglang.launch_server \
  --model-path nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8 \
  --host 0.0.0.0 \
  --port 5000 \
  --trust-remote-code \
  --tp 2 \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nano_v3
```

### NVFP4 (B200)

```shell
python3 -m sglang.launch_server \
  --model-path nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-NVFP4 \
  --host 0.0.0.0 \
  --port 5000 \
  --trust-remote-code \
  --tp 1 \
  --tool-call-parser qwen3_coder \
  --reasoning-parser nano_v3
```


## Generate responses

In [ ]:
## Client Setup
from openai import OpenAI

# The model name used when launching the server.
SERVED_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16"
# SERVED_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8"
# SERVED_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-NVFP4"

BASE_URL = f"http://localhost:5000/v1"
API_KEY = "EMPTY"  # SGLang server doesn't require an API key by default

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print(f"OpenAI client configured to use server at: {BASE_URL}")
print(f"Using model: {SERVED_MODEL_NAME}")

OpenAI client configured to use server at: http://localhost:5000/v1
Using model: nvidia/nemotron-super-sft-020426


### Simple vs streamed generation

In [6]:
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": "Give me 3 bullet points about SGLang."}
    ],
    temperature=0.6,
    max_tokens=512,
)
print("Reasoning:", resp.choices[0].message.reasoning_content, "\nContent:", resp.choices[0].message.content)
print("\n")

Reasoning: The user asks: "Give me 3 bullet points about SGLang." Likely they want concise bullet points summarizing SGLang. SGLang is a system for efficient LLM serving, developed by Stanford, focusing on fast inference with dynamic batching, etc. Provide three bullet points. Ensure correct info. Provide concise bullet points.
 
Content: - **High‑throughput LLM serving**: SGLang is a runtime engine designed to accelerate inference for large language models, delivering up to 10× higher throughput than traditional serving stacks by using optimized kernels and efficient memory management.  

- **Dynamic batching & pipelining**: It supports fine‑grained, request‑level batching and multi‑stage pipelining, allowing the system to continuously pack new queries into GPU work while keeping latency low.  

- **Open‑source & modular**: Built on PyTorch and CUDA, SGLang is released under an MIT license and provides a plug‑and‑play API that can be swapped into existing LLM pipelines (e.g., vLLM, Te

In [7]:
# Streaming chat completion
stream = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"}
    ],
    temperature=0.7,
    max_tokens=1024,
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta
    if delta and delta.content:
        print(delta.content, end="", flush=True)


The first five prime numbers are:

1. **2**
2. **3**
3. **5**
4. **7**
5. **11**

### Reasoning

Note: The model supports two modes - Reasoning ON (default) vs OFF. This can be toggled by setting enable_thinking to False, as shown below. 

In [10]:
# Reasoning on (default)
print("Reasoning on")
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1,
    max_tokens=1024,
)
print(resp.choices[0].message.reasoning_content, resp.choices[0].message.content)
print("\n")
# Reasoning off
print("Reasoning off")
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 facts about SGLang."}
    ],
    temperature=0,
    max_tokens=256,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)
print(resp.choices[0].message.reasoning_content, resp.choices[0].message.content)

Reasoning on
We need to write a haiku (5-7-5 syllable poem) about GPUs. Ensure correct syllable count. Haiku about GPUs. Let's craft:

"Silicon thunder hums,
parallel cores blaze through night's code,
pixels whisper dreams."

Count syllables:

Line1: "Silicon (3) thunder (2) hums (1)" = 6? Actually "Silicon" = 3 (sil-i-con), "thunder" = 2, "hums" =1 => total 6. Need 5. Let's adjust.

Maybe "Silicon hearts beat" = 4? Silicon (3) + hearts (1) =4. Better: "Silicon hearts beat" = 4, need 5. Add "silently"? Let's design.

Try: "Silicon hearts beat" (4) add "still" maybe? "Silicon hearts beat still" Counting: Silicon(3) hearts(1) beat(1) still(1) =6. Hmm.

Let's think: "GPU cores pulse" Count: G-P-U (3?) Actually "GPU" pronounced as three letters, each syllable? Usually "GPU" = "jee-pee-you" three syllables? In spoken haiku, each letter counts as one syllable? Usually "GPU" is 3. So "GPU cores pulse" = GPU(3) cores(1) pulse(1) =5. Good! So line1: "GPU cores pulse". That's 5 syllables.

Line2

### Tool calling

Call functions using the OpenAI Tools schema and inspect returned tool_calls.

In [11]:
# Tool calling via OpenAI tools schema
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate_tip",
            "parameters": {
                "type": "object",
                "properties": {
                    "bill_total": {
                        "type": "integer",
                        "description": "The total amount of the bill"
                    },
                    "tip_percentage": {
                        "type": "integer",
                        "description": "The percentage of tip to be applied"
                    }
                },
                "required": ["bill_total", "tip_percentage"]
            }
        }
    }
]

completion = client.chat.completions.create(
    model="nemotron",
    messages=[
        {"role": "system", "content": ""},
        {"role": "user", "content": "My bill is $50. What will be the amount for 15% tip?"}
    ],
    tools=TOOLS,
    temperature=0.6,
    top_p=0.95,
    max_tokens=512,
    stream=False
)

print(completion.choices[0].message.reasoning_content)
print(completion.choices[0].message.tool_calls)

The user wants to know the tip amount for a $50 bill with a 15% tip. I need to use the calculate_tip function. The function requires bill_total and tip_percentage. Bill total is 50, tip percentage is 15. I'll call the function.

[ChatCompletionMessageFunctionToolCall(id='call_607a46b5f3104184831a7b19', function=Function(arguments='{"bill_total": 50, "tip_percentage": 15}', name='calculate_tip'), type='function', index=0)]


### Controlling Reasoning Budget

The `reasoning_budget` parameter allows you to limit the length of the model's reasoning trace. When the reasoning output reaches the specified token budget, the model will attempt to gracefully end the reasoning at the next newline character. 

If no newline is encountered within 500 tokens after reaching the budget threshold, the reasoning trace will be forcibly terminated at `reasoning_budget + 500` tokens to prevent excessive generation.

In [ ]:
from typing import Any, Dict, List
import openai
from transformers import AutoTokenizer


class ThinkingBudgetClient:
    def __init__(self, base_url: str, api_key: str, tokenizer_name_or_path: str):
        self.base_url = base_url
        self.api_key = api_key
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path)
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)

    def chat_completion(
        self,
        model: str,
        messages: List[Dict[str, Any]],
        reasoning_budget: int = 512,
        max_tokens: int = 1024,
        **kwargs,
    ) -> Dict[str, Any]:
        assert (
            max_tokens > reasoning_budget
        ), f"reasoning_budget must be smaller than max_tokens. Given {max_tokens=} and {reasoning_budget=}"

        # 1. first call chat completion to get reasoning content
        response = self.client.chat.completions.create(
            model=model, 
            messages=messages, 
            max_tokens=reasoning_budget, 
            **kwargs
        )
        
        reasoning_content = response.choices[0].message.reasoning_content or ""
        
        if "</think>" not in reasoning_content:
            # reasoning content is too long, closed with a period (.)
            reasoning_content = f"{reasoning_content}.\n</think>\n\n"
        
        reasoning_tokens_used = len(
            self.tokenizer.encode(reasoning_content, add_special_tokens=False)
        )
        remaining_tokens = max_tokens - reasoning_tokens_used
        
        assert (
            remaining_tokens > 0
        ), f"remaining tokens must be positive. Given {remaining_tokens=}. Increase max_tokens or lower reasoning_budget."

        # 2. append reasoning content to messages and call completion
        messages.append({"role": "assistant", "content": reasoning_content})
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            continue_final_message=True,
        )
        
        response = self.client.completions.create(
            model=model, 
            prompt=prompt, 
            max_tokens=remaining_tokens, 
            **kwargs
        )

        response_data = {
            "reasoning_content": reasoning_content.strip().strip("</think>").strip(),
            "content": response.choices[0].text,
            "finish_reason": response.choices[0].finish_reason,
        }
        return response_data

In [12]:
# Client
SERVED_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16"
# SERVED_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8"
# SERVED_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-NVFP4"
client = ThinkingBudgetClient(
    base_url="http://127.0.0.1:5000/v1",
    api_key="null",
    tokenizer_name_or_path=SERVED_MODEL_NAME
)

In [21]:
resp = client.chat_completion(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1,
    max_tokens=512,
    reasoning_budget=128
)
print("Reasoning:", resp["reasoning_content"], "\nContent:", resp["content"])

Reasoning: The user wants a haiku about GPUs. A haiku is 5-7-5 syllable poem. Must be about GPUs. Provide a haiku. Ensure correct syllable count. Let's craft: "Silicon minds awaken / Teraflops pulse in electric veins / Rendering worlds beyond". Count syllables:

Line1: "Silicon minds awaken" - Si (1) li (2) con (3) minds (4) a (5) wa (6) ken (7) => 7 syllables. Need 5. Let's adjust: "Silicon minds awake" => Si(. 
Content: 
Silicon minds awake  
Teraflops pulse in electric veins  
Worlds rendered in light


## Cleanup and shutdown

To free resources after this notebook:

1. Stop the SGLang server process in the terminal where it was started (`Ctrl+C`).
2. Optionally run the next cell to clear notebook-side CUDA cache.
3. If needed, restart the kernel to ensure a clean state.

In [ ]:
import gc
import torch

# Optional notebook-side cleanup after server shutdown
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Notebook-side CUDA cache cleanup complete.")
print("Primary teardown step: stop the SGLang server with Ctrl+C in its terminal.")

Notebook-side CUDA cache cleanup complete.
Primary teardown step: stop the SGLang server with Ctrl+C in its terminal.
